In [2]:
import gzip
import os

os.chdir("/dss/dsslegfs01/pr53da/pr53da-dss-0018/projects/2020__ancientDNA/05_aDNA/01_angsd_wgs_rescaled_outgroup")
input="wgs_rescaled_outgroup_all_hapconsensus_maxmis_q20"
haplo_file = f'{input}.haplo.gz'
vcf_outfile = f'{input}.vcf.gz'

# Open files
with gzip.open(haplo_file, "rt") as infile, gzip.open(vcf_outfile, "wt") as vcf:
    header = infile.readline().strip().split()
    samples = header[3:]  # get sample names
     # Write minimal VCF header
    vcf.write("##fileformat=VCFv4.2\n")
    vcf.write("##FORMAT=<ID=GT,Number=1,Type=String,Description=\"Genotype\">\n")
    vcf.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\t" + "\t".join(samples) + "\n")

    for line in infile:
        fields = line.strip().split()
        chrom, pos, ref = fields[0], fields[1], fields[2]
        alleles = fields[3:]

        alt_alleles = sorted(set(alleles) - {"N", ref})
        if not alt_alleles:
            continue  # skip invariant/missing

        alt = alt_alleles[0]  # pick first alt allele

        genotypes = []
        for a in alleles:
            if a == "N":
                gt = "./."
            elif a == ref:
                gt = "0/0"
            elif a == alt:
                gt = "1/1"
            else:
                gt = "./."  # ignore unexpected third allele
            genotypes.append(gt)

        vcf.write(f"{chrom}\t{pos}\t.\t{ref}\t{alt}\t.\tPASS\t.\tGT\t" + "\t".join(genotypes) + "\n")